In [1]:
"""
Utilities for Jupyter Notebook integration
"""
import os
import logging
from dotenv import load_dotenv
from utils.config import AppConfig
from utils.logging import configure_logging
from database import SQLiteConnector, PostgresConnector, CSVConnector
from system import TextToSQLSystem

# Default configuration
DEFAULT_CONFIG_DIR = "./config"
DEFAULT_LOG_LEVEL = "INFO"

class NotebookSystem:
    """Simplified interface for notebook testing"""
    
    def __init__(self, datasource="sqlite", llm_provider="groq", config_dir=None):
        """
        Initialize testing system
        
        Args:
            datasource: 'sqlite', 'postgres', or 'csv'
            llm_provider: 'groq', 'openai', 'azure', 'gemini', or 'llama'
            config_dir: Path to config directory
        """
        # Load environment variables
        load_dotenv()
        
        # Setup configuration
        self.config_dir = config_dir or DEFAULT_CONFIG_DIR
        self.config = AppConfig(self.config_dir)
        self.config.system.default_llm_provider = llm_provider
        
        # Configure logging
        self.config.logging.level = DEFAULT_LOG_LEVEL
        configure_logging(self.config.logging)
        
        # Initialize database connector
        if datasource == "sqlite":
            self.db_connector = SQLiteConnector(
                self.config.system, 
                "sample.db"  # Default sample database
            )
        elif datasource == "postgres":
            self.db_connector = PostgresConnector(
                self.config.system,
                host=os.getenv("DB_HOST", "localhost"),
                port=int(os.getenv("DB_PORT", 5432)),
                database=os.getenv("DB_NAME", "testdb"),
                user=os.getenv("DB_USER", "postgres"),
                password=os.getenv("DB_PASSWORD", "")
            )
        elif datasource == "csv":
            self.db_connector = CSVConnector(
                self.config.system,
                "sample.csv",  # Default sample CSV
                "data_table"
            )
        else:
            raise ValueError(f"Unknown datasource: {datasource}")
        
        # Initialize system
        self.system = TextToSQLSystem(self.db_connector, self.config)
    
    def run_query(self, question):
        """
        Execute a natural language query
        
        Args:
            question: Natural language question
            
        Returns:
            dict: Result dictionary with keys:
                - sql: Generated SQL
                - result: DataFrame with results
                - error: Error message if any
                - retries: Number of retries
        """
        return self.system.run_query(question)
    
    def set_llm_provider(self, provider):
        """Dynamically change LLM provider"""
        self.config.system.default_llm_provider = provider
        # Reinitialize system to apply changes
        self.system = TextToSQLSystem(self.db_connector, self.config)
    
    def set_model(self, model):
        """Set specific model for current provider"""
        provider = self.config.system.default_llm_provider
        self.config.llm_config["settings"]["model"] = model
        # Reinitialize system to apply changes
        self.system = TextToSQLSystem(self.db_connector, self.config)
    
    def available_models(self):
        """Return available models for current provider"""
        provider = self.config.system.default_llm_provider
        models = {
            "groq": ["qwen/qwen3-32b"],
            # "openai": ["gpt-3.5-turbo-instruct", "gpt-4", "gpt-4-turbo"],
            # "azure": ["gpt-35-turbo", "gpt-4"],
            # "gemini": ["gemini-pro", "gemini-1.5-pro"],
            # "llama": ["llama2-13b", "llama2-70b"]
        }
        return models.get(provider, [])
    
    def create_sample_data(self):
        """Create sample database with demo data"""
        if isinstance(self.db_connector, SQLiteConnector):
            self.db_connector.create_sample_data()
            print("Created sample SQLite database")
        else:
            print("Sample data creation only supported for SQLite")

In [2]:
import os
load_dotenv()

True

In [3]:
# %ls config

In [4]:
# from configparser import ConfigParser

In [5]:
# config = ConfigParser()

# config.read("C:\\Users\\ASUS\\Documents\\Projects\\agentic-text-to-sql\\config\\prompts.ini")

In [6]:
# config['text_to_sql']['template']

In [7]:
# for key in config['text_to_sql']:
#     print(f"{key}:") 
#     print(f"{config['text_to_sql'][key]['template']}")

In [8]:
# Initialize system
system = NotebookSystem(
    datasource="sqlite",
    llm_provider="groq",
    config_dir="config"  # Path to config folder
)

Loading text-to-SQL prompt template...


In [9]:
# Create sample data
# system.create_sample_data()

In [10]:
# Test Groq with different models
for model in system.available_models():
    print(f"\n{'='*40}")
    print(f"Testing model: {model}")
    system.set_model(model)
    
    result = system.run_query("Show total sales per customer")
    print("SQL:", result["sql"])
    
    if result["result"] is not None and not result["result"].empty:
        print("Result:")
        print(result["result"])
    
    if result["error"]:
        print("Error:", result["error"])


Testing model: qwen/qwen3-32b
Loading text-to-SQL prompt template...
2025-06-21 12:47:32,283 - system - INFO - Processing query: Show total sales per customer...
2025-06-21 12:47:32,286 - system - INFO - Starting workflow execution...
2025-06-21 12:47:32,292 - system - ERROR - Workflow execution failed
Traceback (most recent call last):
  File "c:\Users\ASUS\Documents\Projects\agentic-text-to-sql\system.py", line 28, in run_query
    result = self.workflow.invoke(initial_state.to_dict())
  File "c:\Users\ASUS\Documents\Projects\agentic-text-to-sql\t2sql\lib\site-packages\langgraph\pregel\__init__.py", line 2719, in invoke
    for chunk in self.stream(
  File "c:\Users\ASUS\Documents\Projects\agentic-text-to-sql\t2sql\lib\site-packages\langgraph\pregel\__init__.py", line 2433, in stream
    while loop.tick(input_keys=self.input_channels):
  File "c:\Users\ASUS\Documents\Projects\agentic-text-to-sql\t2sql\lib\site-packages\langgraph\pregel\loop.py", line 553, in tick
    self.tasks = pr

In [11]:
# # Compare providers
# providers = ["groq"]
# questions = [
#     "Who are the top 3 customers by spending?",
#     "Calculate average order value",
#     "Find customers with no orders"
# ]

# for provider in providers:
#     system.set_llm_provider(provider)
#     print(f"\n{'='*40}")
#     print(f"Testing provider: {provider.upper()}")
    
#     for question in questions:
#         result = system.run_query(question)
#         print(f"\nQ: {question}")
#         print(f"SQL: {result['sql']}")
#         print(f"Retries: {result['retries']}")
#         if result["error"]:
#             print(f"Error: {result['error']}")

In [ ]:
from langgraph.graph import StateGraph
from some_module import my_node  # your LangGraph node

builder = StateGraph(input_schema=dict(user_input=str))
builder.add_node("step1", my_node)
builder.set_entry_point("step1")
graph = builder.compile()

output = graph.invoke({"user_input": "Hello"})
print(output)